# 01. Ingest and Deduplicate

This notebook walks through one document end to end with the current `GraphRAG` step API: read, chunk, embed, persist, extract entities, inspect the graph, search it, find duplicate candidates, merge duplicates, and query the cleaned graph again.

Sample data note:

- The tutorial texts in `notebooks/experimental_data/` are Medium.com articles by Filip Wojcik.
- Source profile: `https://medium.com/@filip.igor.wojcik`
- They are fully accessible without a subscription.

Install prerequisites before running the notebook:

```bash
uv sync --group falkordb --extra notebooks
```

You also need model credentials for the chat and embedding models you choose below.


In [1]:
import os
from pathlib import Path

import autoroot  # noqa
from dotenv import load_dotenv
from rich import print

from grawiki.db import FalkorGraphDB
from grawiki.rag import GraphRAG

load_dotenv(override=True)


def locate_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "experimental_data").exists():
        return cwd
    if (cwd / "notebooks" / "experimental_data").exists():
        return cwd / "notebooks"
    raise FileNotFoundError(
        "Could not locate the notebooks directory from the current working directory."
    )


NOTEBOOKS_DIR = locate_notebooks_dir()
DATA_DIR = NOTEBOOKS_DIR / "experimental_data"
DB_PATH = NOTEBOOKS_DIR / "local_falcor.db"
SOURCE_PATH = DATA_DIR / "deep_knowledge_from_deep_learning.txt"
MODEL = (
    os.getenv("GRAWIKI_MODEL")
    or os.getenv("LLM_MODEL")
    or "openrouter/openai/gpt-5-mini"
)
EMBEDDING_MODEL = (
    os.getenv("GRAWIKI_EMBEDDING_MODEL")
    or os.getenv("EMBEDDING_MODEL")
    or "openrouter:openai/text-embedding-3-small"
)

In [2]:
database = FalkorGraphDB(db_path=DB_PATH, graph_name="grawiki_notebooks")
rag = GraphRAG(
    model=MODEL,
    embedding_model=EMBEDDING_MODEL,
    db=database,
    resolve_entities_on_ingest=True,
)

print("GraphRAG initialized with ingest-time entity resolution enabled.")

GraphRAG initialized with ingest-time entity resolution enabled.

## Step-by-step ingestion

The following cell uses the public notebook-friendly methods directly instead of the one-shot `ingest(...)` wrapper. That makes every intermediate artifact visible.


In [3]:
document = rag.read_document(SOURCE_PATH)
chunks = rag.chunk_document(document)
document_embedding = await rag.embed_document(document)
chunk_embeddings = await rag.embed_chunks(chunks)
document_node = rag.build_document_node(document, document_embedding)
chunk_nodes = rag.build_chunk_nodes(chunks, chunk_embeddings)
await rag.persist_document_and_chunks(document_node, chunk_nodes)
chunk_graphs = await rag.extract_kg_per_chunk(chunks)
await rag.persist_entities_and_relationships(
    [chunk.id for chunk in chunks], chunk_graphs
)

print(
    {
        "document_id": document.id,
        "chunk_count": len(chunks),
        "document_embedding_dims": len(document_embedding),
        "chunk_graph_count": len(chunk_graphs),
    }
)

{
    'document_id': '7b4a4df9-f5ba-4b71-bb5a-6ef928fb7357',
    'chunk_count': 5,
    'document_embedding_dims': 0,
    'chunk_graph_count': 5
}

In [4]:
document_rows = database.ro_query(
    "MATCH (n:__document__) RETURN n.id, n.name ORDER BY n.name LIMIT 3"
).result_set
chunk_rows = database.ro_query(
    "MATCH (n:__chunk__) RETURN n.id, n.name, n.document_id ORDER BY n.id LIMIT 5"
).result_set
entity_rows = database.ro_query(
    "MATCH (n:__entity__) RETURN n.id, n.name, n.semantic_key ORDER BY n.semantic_key, n.id LIMIT 10"
).result_set
entities = await database.list_entities()

print({"documents": document_rows, "chunks": chunk_rows, "entity_count": len(entities)})
print("First persisted entities:")
for entity in entities[:10]:
    print(
        {
            "id": entity.id,
            "name": entity.name,
            "labels": sorted(entity.labels),
            "semantic_key": entity.semantic_key,
        }
    )

{
    'documents': [['7b4a4df9-f5ba-4b71-bb5a-6ef928fb7357', 'deep_knowledge_from_deep_learning']],
    'chunks': [
        [
            '538da7a4-b293-4fa4-b2f6-a6d0a3cf8015',
            'Chunk 538da7a4-b293-4fa4-b2f6-a6d0a3cf8015',
            '7b4a4df9-f5ba-4b71-bb5a-6ef928fb7357'
        ],
        [
            '623b14e1-973e-44e0-a478-3545ac98d724',
            'Chunk 623b14e1-973e-44e0-a478-3545ac98d724',
            '7b4a4df9-f5ba-4b71-bb5a-6ef928fb7357'
        ],
        [
            '7ef7deb6-79bd-4629-99e6-81d5739db015',
            'Chunk 7ef7deb6-79bd-4629-99e6-81d5739db015',
            '7b4a4df9-f5ba-4b71-bb5a-6ef928fb7357'
        ],
        [
            'a1057570-4177-4050-b3d6-29c8531e2604',
            'Chunk a1057570-4177-4050-b3d6-29c8531e2604',
            '7b4a4df9-f5ba-4b71-bb5a-6ef928fb7357'
        ],
        [
            'e9d4e3ae-2b09-4d47-b77c-7c8022979c36',
            'Chunk e9d4e3ae-2b09-4d47-b77c-7c8022979c36',
            '7b4a4df9-f5ba-4b71-bb5a-6ef928fb7357'
        ]
    ],
    'entity_count': 74
}

First persisted entities:

{
    'id': '3dff7163-0356-4789-ae16-160c9c7bf618',
    'name': 'Researchers',
    'labels': ['Agent'],
    'semantic_key': 'Agent_researchers'
}

{
    'id': '6de99fac-f53b-43ef-a63c-c56e94034907',
    'name': 'Indirect question answering',
    'labels': ['Application'],
    'semantic_key': 'Application_indirect-question-answering'
}

{
    'id': 'b2755b1d-f105-4426-9f49-baaba9a8a488',
    'name': 'This post / planned series',
    'labels': ['Blog'],
    'semantic_key': 'Blog_first-post-series'
}

...

{
    'id': '17e19811-6285-475a-ad44-8968019701f0',
    'name': 'Software for building KGs',
    'labels': ['Concept'],
    'semantic_key': 'Concept_kg-software'
}

{
    'id': 'c046a961-296d-4409-9418-caecfec91a9c',
    'name': 'KG reasoning (KGR)',
    'labels': ['Concept'],
    'semantic_key': 'Concept_kgr'
}

{
    'id': 'dd2ac635-778b-4435-b75b-6a39a5002758',
    'name': 'Knowledge graphs (KG)',
    'labels': ['Concept'],
    'semantic_key': 'Concept_knowledge-graphs'
}

{
    'id': '17d16d9c-2359-4477-abf8-911c70adc64f',
    'name': 'Large Language Models (LLMs)',
    'labels': ['Concept'],
    'semantic_key': 'Concept_llms'
}

{
    'id': '6e4e5e46-e278-4b29-be8d-7e07d2d285e0',
    'name': 'Probabilistic reasoning',
    'labels': ['Concept'],
    'semantic_key': 'Concept_probabilistic-reasoning'
}

In [5]:
focus_entity = entities[0] if entities else None
if focus_entity is None:
    print("No entities were persisted.")
else:
    neighbors = await database.neighbor_relationships(
        node_ids=[focus_entity.id], limit_per_node=5
    )
    print({"focus_entity": focus_entity.name, "neighbors": neighbors[focus_entity.id]})

search_hits = await rag.search("Knowledge graphs", limit=5)
print("GraphRAG search hits:")
for hit in search_hits:
    print(
        {
            "score": round(hit.score, 4),
            "matched_on": hit.matched_on,
            "id": hit.node.id,
            "name": hit.node.name,
            "labels": sorted(hit.node.labels),
        }
    )

{
    'focus_entity': 'Researchers',
    'neighbors': [
        NeighborRelationship(
            source_id='3dff7163-0356-4789-ae16-160c9c7bf618',
            source_name='Researchers',
            relationship_label='__mentions__',
            target=ChunkNode(
                id='538da7a4-b293-4fa4-b2f6-a6d0a3cf8015',
                labels=frozenset({'__chunk__'}),
                semantic_key='chunk_538da7a4-b293-4fa4-b2f6-a6d0a3cf8015',
                name='Chunk 538da7a4-b293-4fa4-b2f6-a6d0a3cf8015',
                properties={},
                embedding=[],
                document_id='7b4a4df9-f5ba-4b71-bb5a-6ef928fb7357',
                content='Introduction\nThe rise of ChatGPT and other Large Language Models (LLMs) has raised many 
questions about how reliable the knowledge that such tools possess and use is. Language models of any type or 
family are tools that can generate texts that are the result of probabilistic reasoning over the corpus that they 
were trained on. On their own, if not reinforced with additional plugins that allow them to browse the Web or read 
external sources via context injection, they can hallucinate or perform counterfactual reasoning.\n\nMany 
researchers suggest, that enriching the LLMs training process and prompt context with knowledge graphs (KG) and 
reasoning over them might lead to the combination of the best properties of both types of systems ([1], 
[2]).\n\nThis is the first post from the planned longer series, that will present the idea of knowledge graphs, KG 
reasoning (KGR), graph neural networks, and their application to indirect question answering.\n\nAfter reading this
post, you will learn\n\nWhat is a heterogeneous graph, and how is it different from typical graph structures?\nWhat
is a knowledge graph, and what are its main parts?\nWhat kind of software allows you to build knowledge graphs and 
later use them to train graph neural networks?\nHow to construct a heterogeneous graph in Neo4j and PyTorch 
Geometric, using the data extracted from semantic source like DbPedia.\nWhat are some notable use cases for 
knowledge graphs?\nHeterogeneous graphs\nLet’s start with the basics. What is a heterogeneous graph (HG)? This 
mysterious name describes a graph in which we have various types of edges (that depict specific entities) and 
vertices, that depict relations.\n\nFormally, heterogeneous graphs are defined as follows 
([4]):\n\n[IMAGE]\n\nWhile this equation looks complicated, what it really says, is that each node has a specific 
type assigned to it, and every edge represents some relation.\n\nOne picture says more than a thousand words, so 
let’s take a look at the following picture. ',
                metadata={},
                doc_position=0
            )
        ),
        NeighborRelationship(
            source_id='3dff7163-0356-4789-ae16-160c9c7bf618',
            source_name='Researchers',
            relationship_label='suggest_enriching_with',
            target=Node(
                id='dd2ac635-778b-4435-b75b-6a39a5002758',
                labels=frozenset({'Concept'}),
                semantic_key='Concept_knowledge-graphs',
                name='Knowledge graphs (KG)',
                properties={},
                embedding=[]
            )
        )
    ]
}

GraphRAG search hits:

{
    'score': 0.9378,
    'matched_on': 'keyword_path',
    'id': '8902e471-d6df-4d1e-8991-227f6a234024',
    'name': 'Knowledge Graph',
    'labels': ['Concept']
}

{
    'score': 0.9007,
    'matched_on': 'keyword_path',
    'id': 'dd2ac635-778b-4435-b75b-6a39a5002758',
    'name': 'Knowledge graphs (KG)',
    'labels': ['Concept']
}

{
    'score': 0.881,
    'matched_on': 'keyword_path',
    'id': '3355e95c-7ff9-4908-8f43-f216a552981a',
    'name': 'knowledge graph',
    'labels': ['Concept']
}

{
    'score': 0.6594,
    'matched_on': 'keyword_path',
    'id': '163b886c-dc57-4c02-b87b-8c624a7fcb33',
    'name': 'Open Knowledge Graph (OKG)',
    'labels': ['Concept']
}

{
    'score': 0.6491,
    'matched_on': 'vector',
    'id': '7ef7deb6-79bd-4629-99e6-81d5739db015',
    'name': 'Chunk 7ef7deb6-79bd-4629-99e6-81d5739db015',
    'labels': ['__chunk__']
}

## Duplicate inspection

Use the duplicate finder first so the notebook shows the candidate groups before applying the destructive merge step. Exact counts may vary slightly across model runs.


In [6]:
def merge_report_payload(report):
    return {
        "master_id": report.master_id,
        "duplicate_ids": list(report.duplicate_ids),
        "source": report.source,
        "merged_labels": list(report.merged_labels),
        "property_conflicts": report.property_conflicts,
    }


duplicate_candidates = await rag.find_entity_duplicate_candidates(
    limit=5, threshold=0.9
)
dry_run_reports = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.95, dry_run=True
)

print(
    {
        "semantic_key_collision_groups": len(
            duplicate_candidates.semantic_key_collision_candidates
        ),
        "similarity_candidate_groups": len(duplicate_candidates.similarity_candidates),
        "dry_run_merge_reports": [
            merge_report_payload(report) for report in dry_run_reports
        ],
    }
)

{
    'semantic_key_collision_groups': 0,
    'similarity_candidate_groups': 12,
    'dry_run_merge_reports': [
        {
            'master_id': '55da1c43-b185-4586-905c-7d441758615b',
            'duplicate_ids': ['7fad0443-8a87-43ee-9ed3-5027b185833e', '0d3adfee-40d8-4fed-b3ee-c1de18313192'],
            'source': 'similarity',
            'merged_labels': ['Concept'],
            'property_conflicts': ()
        },
        {
            'master_id': '11dd0dd7-b9e9-4d00-baa3-1623a18d98bb',
            'duplicate_ids': ['7edffe80-a265-47b7-80ef-3b5ed8795319', 'a7139284-848c-44ff-a562-5c1ff28dfd4c'],
            'source': 'similarity',
            'merged_labels': ['Dataset', 'Project'],
            'property_conflicts': ()
        },
        {
            'master_id': '2e7d9782-5440-4a04-bafb-9fdbe3b53759',
            'duplicate_ids': ['3560f4ee-80e9-4d0d-8e03-37d5a5900d75', '6d03b3c5-adf3-460d-92df-e483599325f5'],
            'source': 'similarity',
            'merged_labels': ['Library', 'Software', 'Tool'],
            'property_conflicts': ()
        },
        {
            'master_id': 'e4aa99e4-f82e-4d72-a6b9-a34bcd5c5a38',
            'duplicate_ids': ['c87ec764-2b9d-4d56-b3ee-5364bff55077', '2c2c8d1e-155f-415f-a0cb-479384afeb95'],
            'source': 'similarity',
            'merged_labels': ['Software', 'Tool'],
            'property_conflicts': ()
        },
        {
            'master_id': 'c2c05698-e9cf-43e5-8c38-bec26f87e264',
            'duplicate_ids': ['d655cb2b-b7d0-4eea-bb4b-4cd482a6c93f'],
            'source': 'similarity',
            'merged_labels': ['Concept', 'Technology'],
            'property_conflicts': ()
        },
        {
            'master_id': '6729a50d-6672-4ec2-9787-cac70d72cd8e',
            'duplicate_ids': ['d6eb14d6-4022-43aa-bdd9-d2fd1f679659'],
            'source': 'similarity',
            'merged_labels': ['QueryLanguage', 'Technology'],
            'property_conflicts': ()
        }
    ]
}

In [7]:
applied_reports = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.95, dry_run=False
)
entities_after_dedupe = await database.list_entities()

print("Applied merge reports:")
for report in applied_reports:
    print(merge_report_payload(report))

print({"entity_count_after_dedupe": len(entities_after_dedupe)})
for entity in entities_after_dedupe[:10]:
    print(
        {
            "id": entity.id,
            "name": entity.name,
            "labels": sorted(entity.labels),
            "semantic_key": entity.semantic_key,
        }
    )

Applied merge reports:

{
    'master_id': '55da1c43-b185-4586-905c-7d441758615b',
    'duplicate_ids': ['7fad0443-8a87-43ee-9ed3-5027b185833e', '0d3adfee-40d8-4fed-b3ee-c1de18313192'],
    'source': 'similarity',
    'merged_labels': ['Concept'],
    'property_conflicts': ()
}

{
    'master_id': '11dd0dd7-b9e9-4d00-baa3-1623a18d98bb',
    'duplicate_ids': ['7edffe80-a265-47b7-80ef-3b5ed8795319', 'a7139284-848c-44ff-a562-5c1ff28dfd4c'],
    'source': 'similarity',
    'merged_labels': ['Dataset', 'Project'],
    'property_conflicts': ()
}

{
    'master_id': '2e7d9782-5440-4a04-bafb-9fdbe3b53759',
    'duplicate_ids': ['3560f4ee-80e9-4d0d-8e03-37d5a5900d75', '6d03b3c5-adf3-460d-92df-e483599325f5'],
    'source': 'similarity',
    'merged_labels': ['Library', 'Software', 'Tool'],
    'property_conflicts': ()
}

{
    'master_id': 'e4aa99e4-f82e-4d72-a6b9-a34bcd5c5a38',
    'duplicate_ids': ['c87ec764-2b9d-4d56-b3ee-5364bff55077', '2c2c8d1e-155f-415f-a0cb-479384afeb95'],
    'source': 'similarity',
    'merged_labels': ['Software', 'Tool'],
    'property_conflicts': ()
}

...

{
    'id': 'c046a961-296d-4409-9418-caecfec91a9c',
    'name': 'KG reasoning (KGR)',
    'labels': ['Concept'],
    'semantic_key': 'Concept_kgr'
}

{
    'id': 'dd2ac635-778b-4435-b75b-6a39a5002758',
    'name': 'Knowledge graphs (KG)',
    'labels': ['Concept'],
    'semantic_key': 'Concept_knowledge-graphs'
}

{
    'id': '17d16d9c-2359-4477-abf8-911c70adc64f',
    'name': 'Large Language Models (LLMs)',
    'labels': ['Concept'],
    'semantic_key': 'Concept_llms'
}

{
    'id': '6e4e5e46-e278-4b29-be8d-7e07d2d285e0',
    'name': 'Probabilistic reasoning',
    'labels': ['Concept'],
    'semantic_key': 'Concept_probabilistic-reasoning'
}

{
    'id': '1fd17632-e65f-4692-bea0-424281d8519e',
    'name': 'Training corpus',
    'labels': ['Data'],
    'semantic_key': 'Data_training-corpus'
}

In [8]:
final_hits = await rag.search("Knowledge graph concept", limit=5)
comparison_rows = database.ro_query(
    "MATCH (s:__entity__)-[r]->(t:__entity__) RETURN s.name, type(r), t.name ORDER BY s.name, type(r), t.name LIMIT 10"
).result_set

print("Final GraphRAG hits:")
for hit in final_hits:
    print(
        {
            "score": round(hit.score, 4),
            "matched_on": hit.matched_on,
            "name": hit.node.name,
            "labels": sorted(hit.node.labels),
            "content_preview": hit.node.properties.get("content", "")[:500],
        }
    )

print({"entity_relationship_samples": comparison_rows})

Final GraphRAG hits:

{
    'score': 1.0,
    'matched_on': 'keyword_path',
    'name': 'knowledge graph',
    'labels': ['Concept'],
    'content_preview': 'Source Node: knowledge graph (id: 3355e95c-7ff9-4908-8f43-f216a552981a), similarity: 
1.0000\n-[__mentions__]-> NAME: Chunk a1057570-4177-4050-b3d6-29c8531e2604; LABELS: __chunk__; CONTENT: We have the
following node types: “author”, “paper”, “institution”, and the following relation types: “wrote”, “published”, 
“cooperated”.\n\nI believe that this is much more intuitive than the formal expression above.\n\nAnother variation 
of heterogeneous graphs (or their subtype) is the so-called “property graph” '
}

{
    'score': 0.9294,
    'matched_on': 'keyword_path',
    'name': 'Knowledge Graph',
    'labels': ['Concept'],
    'content_preview': 'Source Node: Knowledge Graph (id: 8902e471-d6df-4d1e-8991-227f6a234024), similarity: 
0.9294\n-[__mentions__]-> NAME: Chunk 623b14e1-973e-44e0-a478-3545ac98d724; LABELS: __chunk__; CONTENT: You will 
see, how it can efficiently represent multidimensional data, allowing researchers and analysts to get useful 
insights.\n\nIn this case study, we will show how knowledge graphs may effectively capture and analyse complicated 
connections. We will construct a heterogeneous graph that contains selected chara'
}

{
    'score': 0.8137,
    'matched_on': 'keyword_path',
    'name': 'Knowledge graphs (KG)',
    'labels': ['Concept'],
    'content_preview': 'Source Node: Knowledge graphs (KG) (id: dd2ac635-778b-4435-b75b-6a39a5002758), similarity: 
0.8137\n-[__mentions__]-> NAME: Chunk 538da7a4-b293-4fa4-b2f6-a6d0a3cf8015; LABELS: __chunk__; CONTENT: 
Introduction\nThe rise of ChatGPT and other Large Language Models (LLMs) has raised many questions about how 
reliable the knowledge that such tools possess and use is. Language models of any type or family are tools that can
generate texts that are the result of probabilistic reasoning over the corpus that'
}

{
    'score': 0.6639,
    'matched_on': 'vector',
    'name': 'Chunk 7ef7deb6-79bd-4629-99e6-81d5739db015',
    'labels': ['__chunk__'],
    'content_preview': ''
}

{
    'score': 0.6458,
    'matched_on': 'keyword_path',
    'name': 'Open Knowledge Graph (OKG)',
    'labels': ['Concept'],
    'content_preview': 'Source Node: Open Knowledge Graph (OKG) (id: 163b886c-dc57-4c02-b87b-8c624a7fcb33), 
similarity: 0.6458\n-[__mentions__]-> NAME: Chunk 7ef7deb6-79bd-4629-99e6-81d5739db015; LABELS: __chunk__; CONTENT:
A knowledge graph (i) mainly describes real world entities and their interrelations, organized in a graph, (ii) 
defines possible classes and relations of entities in a schema, (iii) allows for potentially interrelating 
arbitrary entities with each other and (iv) covers various topical domains [6].\n\nK'
}

{
    'entity_relationship_samples': [
        ['Case Study', 'constructs', 'heterogeneous graph'],
        ['House', 'make_up', 'Legislature'],
        ['Knowledge Graph', 'defines', 'Schema'],
        ['Knowledge Graph', 'describes', 'Real-world entities'],
        ['Knowledge Graph', 'represents', 'Multidimensional Data'],
        ['Large Language Models (LLMs)', 'can_produce', 'Hallucination'],
        ['Large Language Models (LLMs)', 'perform', 'Probabilistic reasoning'],
        ['Legislature', 'belongs_to', 'Nation'],
        ['Multinational Organisations', 'includes', 'European Union'],
        ['Multinational Organisations', 'includes', 'NATO']
    ]
}

## Next steps

- Notebook 2 reuses the same FalkorDB graph for a `pydantic_ai.Agent` demo with `search`, `remember`, and `recall`.
- Notebook 3 builds a lightweight visualization over the populated graph.
- The other article files in `experimental_data/` are good follow-up ingestion exercises once this baseline flow works.


In [9]:
# Run this once when you are finished with the notebook session.
database.close()